# Propriétés

Une *propriété* est un attribut dont la lecture, l'écriture et la suppression passent par des méthodes, tout en gardant la syntaxe d'un simple attribut (sans parenthèses).

Ces travaux pratiques partent de code client qu'il ne faut pas changer, puis vont vers le mécanisme des propriétés. Chaque exercice fournit la classe dont il part : vous pouvez les faire dans l'ordre que vous voulez, et le dernier est facultatif.

## Valider sans changer le code client

La classe `Compte` ci-dessous expose un attribut public `solde`, et la cellule qui la suit représente du code client, écrit depuis longtemps et utilisé partout.

Nouveau besoin : un solde ne peut plus être négatif. Modifiez `Compte` pour que toute écriture d'un solde négatif lève une `ValueError`, à la construction comme ensuite, sans changer une seule ligne du code client.

Une fois la classe modifiée, relancez le code client : il doit afficher `150`, deux messages d'erreur, puis de nouveau `150`.

In [ ]:
class Compte:
  def __init__(self, solde):
    self.solde = solde

  # Votre code ici

In [ ]:
compte = Compte(100)
compte.solde += 50
print(compte.solde)

try:
  compte.solde = -1
except ValueError as erreur:
  print(f"Erreur : {erreur}")

try:
  Compte(-1)
except ValueError as erreur:
  print(f"Erreur : {erreur}")

print(compte.solde)

### Conseils supplémentaires

- `solde` devient une propriété : un getter sous `@property`, un setter sous `@solde.setter`, tous deux nommés `solde`.
- La valeur va dans un attribut interne d'un autre nom, `_solde`. Écrire `self.solde` dans le setter le rappellerait sans fin.
- `__init__` ne change pas : `self.solde = solde` passe déjà par le setter.

```python
class Compte:
  def __init__(self, solde):
    self.solde = solde

  @property
  def solde(self):
    return ...

  @solde.setter
  def solde(self, valeur):
    if ...:
      raise ValueError("Le solde ne peut pas être négatif")
    self._solde = ...
```

### Solution

In [ ]:
class Compte:
  def __init__(self, solde):
    self.solde = solde

  @property
  def solde(self):
    return self._solde

  @solde.setter
  def solde(self, valeur):
    if valeur < 0:
      raise ValueError("Le solde ne peut pas être négatif")
    self._solde = valeur


compte = Compte(100)
compte.solde += 50
print(compte.solde)

try:
  compte.solde = -1
except ValueError as erreur:
  print(f"Erreur : {erreur}")

try:
  Compte(-1)
except ValueError as erreur:
  print(f"Erreur : {erreur}")

print(compte.solde)

## Propriété en lecture seule

Ajoutez à la classe `Cercle` une propriété `aire`, en lecture seule (sans setter), qui calcule l'aire du disque à partir de `rayon` (utilisez `math.pi`). `Cercle(5).aire` doit valoir environ `78.54`.

In [ ]:
import math


class Cercle:
  def __init__(self, rayon):
    self.rayon = rayon

  @property
  def rayon(self):
    return self._rayon

  @rayon.setter
  def rayon(self, valeur):
    if valeur <= 0:
      raise ValueError("Le rayon doit être strictement positif")
    self._rayon = valeur

  # Votre code ici

Que se passe-t-il si vous exécutez `c.aire = 10` ? Essayez avant de lire la solution.

*Votre réponse*

### Conseils supplémentaires

Une méthode sous `@property`, sans `@aire.setter`. Elle lit `self.rayon` et renvoie $\pi r^2$.

```python
@property
def aire(self):
  return ...
```

### Solution

Une propriété sans setter est en lecture seule : toute tentative d'affectation lève une `AttributeError`.

En interne, `property` est un *descripteur de données* : il définit `__set__`, même sans setter (ce `__set__` lève alors l'erreur), ce qui lui donne priorité sur le `__dict__` de l'instance.

In [ ]:
import math


class Cercle:
  def __init__(self, rayon):
    self.rayon = rayon

  @property
  def rayon(self):
    return self._rayon

  @rayon.setter
  def rayon(self, valeur):
    if valeur <= 0:
      raise ValueError("Le rayon doit être strictement positif")
    self._rayon = valeur

  @property
  def aire(self):
    return math.pi * self.rayon**2


c = Cercle(5)
print(c.aire)

try:
  c.aire = 10
except AttributeError as erreur:
  print(f"Erreur : {erreur}")

## Invalider un cache

Le calcul de `aire` est ici trivial, mais imaginons qu'il soit long, comme une intégration numérique sur une grille très fine. La classe ci-dessous le met en cache avec `functools.cached_property`.

Exécutez-la, puis la cellule qui la suit : après `c.rayon = 10`, l'aire affichée est périmée.

1. Modifiez le setter de `rayon` pour qu'il invalide le cache : `cached_property` range la valeur dans le dictionnaire de l'instance, `self.__dict__`, sous le nom `"aire"`. La cellule suivante doit alors afficher `Calcul de l'aire...` deux fois, et une aire d'environ `314.16` à la fin.
2. Ajoutez un deleter à `rayon`, qui supprime `_rayon` et le cache : après `del c.rayon`, `vars(c)` doit valoir `{}`.

**Documentation utile**

- [`functools.cached_property`](https://docs.python.org/fr/3/library/functools.html#functools.cached_property)
- [`dict.pop`](https://docs.python.org/fr/3/library/stdtypes.html#dict.pop)
- [`property.deleter`](https://docs.python.org/fr/3/library/functions.html#property.deleter)

In [ ]:
import functools
import math


class Cercle:
  def __init__(self, rayon):
    self.rayon = rayon

  @property
  def rayon(self):
    return self._rayon

  @rayon.setter
  def rayon(self, valeur):
    if valeur <= 0:
      raise ValueError("Le rayon doit être strictement positif")
    self._rayon = valeur
    # Votre code ici

  @functools.cached_property
  def aire(self):
    print("Calcul de l'aire...")
    return math.pi * self.rayon**2

In [ ]:
c = Cercle(5)
print(c.aire)
print(c.aire)
c.rayon = 10
print(c.aire)

### Conseils supplémentaires

- `self.__dict__.pop("aire", None)` retire la valeur en cache. Sans le `None`, le constructeur planterait : `__init__` passe par le setter alors que rien n'est encore en cache, et `pop` lèverait une `KeyError`.
- Le deleter s'écrit sous `@rayon.deleter`, avec le nom `rayon` lui aussi.

```python
@rayon.setter
def rayon(self, valeur):
  if valeur <= 0:
    raise ValueError("Le rayon doit être strictement positif")
  self._rayon = valeur
  self.__dict__.pop(...)


@rayon.deleter
def rayon(self):
  del self._rayon
  self.__dict__.pop(...)
```

### Solution

In [ ]:
import functools
import math


class Cercle:
  def __init__(self, rayon):
    self.rayon = rayon

  @property
  def rayon(self):
    return self._rayon

  @rayon.setter
  def rayon(self, valeur):
    if valeur <= 0:
      raise ValueError("Le rayon doit être strictement positif")
    self._rayon = valeur
    self.__dict__.pop("aire", None)

  @rayon.deleter
  def rayon(self):
    del self._rayon
    self.__dict__.pop("aire", None)

  @functools.cached_property
  def aire(self):
    print("Calcul de l'aire...")
    return math.pi * self.rayon**2


c = Cercle(5)
print(c.aire)
print(c.aire)
c.rayon = 10
print(c.aire)
del c.rayon
print(vars(c))

`cached_property` demande que les instances possèdent un `__dict__` : c'est le cas par défaut, sauf si la classe définit `__slots__` sans `"__dict__"`.

## Ce que fait `@x.setter`

`@property` n'est que du sucre syntaxique, et `@rayon.setter` aussi. Pour voir ce qu'il fait, écrivez une classe `Cercle` sans aucun `@` : le getter s'appelle `lire`, le setter `ecrire` (avec la même validation que plus haut), et la propriété `rayon` se construit en deux affectations, d'abord `property` avec le getter seul, puis la méthode `setter` de cette propriété.

Résultat attendu : `Cercle(5).rayon` vaut `5`, `Cercle.rayon.fget.__name__` vaut `"lire"` et `Cercle.rayon.fset.__name__` vaut `"ecrire"`.

**Documentation utile**

- [`property`](https://docs.python.org/fr/3/library/functions.html#property)
- [`property.setter`](https://docs.python.org/fr/3/library/functions.html#property.setter)

In [ ]:
# Votre code ici

La méthode `setter` modifie-t-elle la propriété, ou en renvoie-t-elle une autre ? Pourquoi, alors, les deux méthodes portent-elles le même nom quand on écrit `@rayon.setter` ?

*Votre réponse*

### Conseils supplémentaires

Les deux affectations se font dans le corps de la classe, après les deux méthodes. Pour répondre à la question, comparez avec `is` une propriété et ce que renvoie sa méthode `setter`.

```python
class Cercle:
  def __init__(self, rayon):
    self.rayon = rayon

  def lire(self):
    return self._rayon

  def ecrire(self, valeur): ...

  rayon = property(...)
  rayon = rayon.setter(...)
```

### Solution

In [ ]:
class Cercle:
  def __init__(self, rayon):
    self.rayon = rayon

  def lire(self):
    return self._rayon

  def ecrire(self, valeur):
    if valeur <= 0:
      raise ValueError("Le rayon doit être strictement positif")
    self._rayon = valeur

  rayon = property(lire)
  rayon = rayon.setter(ecrire)


print(Cercle(5).rayon)
print(Cercle.rayon.fget.__name__, Cercle.rayon.fset.__name__)

sans_setter = property(Cercle.lire)
avec_setter = sans_setter.setter(Cercle.ecrire)
print(sans_setter is avec_setter, sans_setter.fset)

`setter` ne modifie pas la propriété : il en renvoie une nouvelle, qui a en plus le setter. Il faut donc ranger le résultat sous le nom `rayon`.

C'est ce que fait `@rayon.setter` au-dessus de `def rayon` : le résultat du décorateur prend le nom de la méthode. Si la méthode s'appelle `ecrire`, c'est `ecrire` qui reçoit la nouvelle propriété, et `rayon` reste sans setter.

## Pour aller plus loin : un descripteur réutilisable

Cet exercice est facultatif.

Une `Enceinte` a un `volume` entre 0 et 100 et des `basses` entre -10 et 10. Avec des propriétés, il faudrait un getter et un setter par attribut. Complétez plutôt le descripteur `Entre`, qui valide n'importe quel attribut entre deux bornes : ajoutez-lui `__set_name__`, `__get__` et `__set__`.

Une fois le descripteur écrit, la cellule d'essai doit afficher `30 80`, un message d'erreur qui nomme `volume`, puis `30` : deux enceintes ne partagent pas leurs valeurs, et une valeur refusée ne change rien.

In [ ]:
class Entre:
  def __init__(self, minimum, maximum):
    self.minimum = minimum
    self.maximum = maximum

  # Votre code ici


class Enceinte:
  volume = Entre(0, 100)
  basses = Entre(-10, 10)

  def __init__(self, volume, basses):
    self.volume = volume
    self.basses = basses

In [ ]:
salon = Enceinte(30, 2)
cuisine = Enceinte(80, -5)
print(salon.volume, cuisine.volume)

try:
  salon.volume = 150
except ValueError as erreur:
  print(f"Erreur : {erreur}")

print(salon.volume)

### Conseils supplémentaires

- `__set_name__(self, proprietaire, nom)` est appelé à la création de la classe, une fois par attribut : il donne le nom de l'attribut (`"volume"`).
- Un seul `Entre` sert toutes les instances d'`Enceinte` : la valeur ne va pas dans `self`, mais dans `objet`, avec `setattr` et `getattr`, sous un nom interne (`"_volume"`).
- Lu sur la classe (`Enceinte.volume`), `__get__` reçoit `None` pour `objet` : il renvoie alors le descripteur lui-même.

```python
class Entre:
  def __init__(self, minimum, maximum):
    self.minimum = minimum
    self.maximum = maximum

  def __set_name__(self, proprietaire, nom):
    self.nom_prive = ...

  def __get__(self, objet, proprietaire=None):
    if objet is None:
      return self
    return getattr(...)

  def __set__(self, objet, valeur):
    if not ...:
      raise ValueError(...)
    setattr(...)
```

### Solution

In [ ]:
class Entre:
  def __init__(self, minimum, maximum):
    self.minimum = minimum
    self.maximum = maximum

  def __set_name__(self, proprietaire, nom):
    self.nom_prive = "_" + nom

  def __get__(self, objet, proprietaire=None):
    if objet is None:
      return self
    return getattr(objet, self.nom_prive)

  def __set__(self, objet, valeur):
    if not self.minimum <= valeur <= self.maximum:
      raise ValueError(
        f"{self.nom_prive[1:]} doit être entre {self.minimum} et {self.maximum}"
      )
    setattr(objet, self.nom_prive, valeur)


class Enceinte:
  volume = Entre(0, 100)
  basses = Entre(-10, 10)

  def __init__(self, volume, basses):
    self.volume = volume
    self.basses = basses


salon = Enceinte(30, 2)
cuisine = Enceinte(80, -5)
print(salon.volume, cuisine.volume)

try:
  salon.volume = 150
except ValueError as erreur:
  print(f"Erreur : {erreur}")

print(salon.volume)
print(vars(salon))